# Translate Ensembl IDs to Gene Symbols
1) Translates the Ensembl IDs in rpkm_combined.txt to Gene Symbols
2) Labels all samples with a condition row (healthy, t2d)

In [27]:
import os
import numpy as np
import pandas as pd
import json

import mygene

rpkm = r"../rpkm_values/rpkm_combined.txt"
healthy = r"../rpkm_values/rpkm_healthy/rpkm_values_healthy.txt"
t2d = r"../rpkm_values/rpkm_t2d/rpkm_values_t2d.txt"

In [28]:
df_combined = pd.read_csv(rpkm, sep='\t')
df_healthy = pd.read_csv(healthy, sep='\t')
df_t2d = pd.read_csv(t2d, sep='\t')

In [29]:
# Transpose the RPKM dataframe so rows become samples
df = df_combined.T

# Now determine sample labels
healthy_samples = set(df_healthy.columns)
t2d_samples = set(df_t2d.columns)

# Assign condition based on sample name
df['condition'] = ['healthy' if sample in healthy_samples else 't2d' for sample in df.index]


In [30]:
# Create a dictionary mapping ensembl_ids to gene symbols

ensembl_ids = list(df.drop(columns=['condition']).columns)

query_cache_file = "query_result.json"

# Batch query: return gene symbol for each Ensembl ID (This will take 2 minutes)
if os.path.exists(query_cache_file):
    with open(query_cache_file, "r") as f:
        query_result = json.load(f)
    print(f"Loaded cached MyGene.info results from {query_cache_file}")
else:
    mg = mygene.MyGeneInfo()
    query_result = mg.querymany(ensembl_ids, scopes='ensembl.gene', fields='symbol', species='human')

    # Save for future reuse
    with open(query_cache_file, "w") as f:
        json.dump(query_result, f)
    print(f"Fetched MyGene.info results and cached to {query_cache_file}")

# Build a mapping: ENSG ID → gene symbol
ensembl_to_symbol = {
    item['query']: item.get('symbol', item['query'])  # fallback to ENSG ID if no symbol found
    for item in query_result
}

no_hit_count = sum(1 for item in query_result if item.get('notfound', False))
print(f"{len(ensembl_to_symbol) - no_hit_count} of {len(ensembl_ids)} ensembl to symbol mapped")

Loaded cached MyGene.info results from query_result.json
61474 of 62710 ensembl to symbol mapped


In [31]:
# Rename columns from ensembl ids to gene ids
df.rename(columns=ensembl_to_symbol, inplace=True)
gene_ids = list(df.drop(columns=['condition']).columns)
len(gene_ids)

62710

In [32]:
df_out = df.T

In [33]:
# Step 2: Extract condition row and drop from df_out
condition_row = df_out.loc['condition']
df_out = df_out.drop(index='condition')

# Step 3: Prepend the condition row
df_out_with_condition = pd.concat([condition_row.to_frame().T, df_out])

In [34]:
df_out_with_condition.head()

,AZ_A8.bam,AZ_C10.bam,AZ_C4.bam,AZ_D6.bam,AZ_E7.bam,AZ_F1.bam,AZ_F2.bam,AZ_H11.bam,HP1502401_A10.bam,HP1502401_A6.bam,...,HP1526901T2D_K7.bam,HP1526901T2D_L1.bam,HP1526901T2D_L5.bam,HP1526901T2D_M15.bam,HP1526901T2D_N1.bam,HP1526901T2D_N12.bam,HP1526901T2D_N13.bam,HP1526901T2D_N23.bam,HP1526901T2D_N3.bam,HP1526901T2D_O14.bam
condition,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,...,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d
ATAD3B,0.0,0.0,0.0,0.0,0.073673,0.0,0.0,0.0,0.0,0.0,...,0.767416,0.0,3.377254,0.881244,1.27467,0.0,0.0,0.0,0.23512,0.0
DDX11L17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000228037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PRDM16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.083458,0.0,...,0.0,3.525674,3.454986,0.0,0.0,0.0,1.013528,0.0,0.0,2.307148


In [35]:
# df_out_with_condition.to_csv("rpkm_combined_labeled.txt", sep='\t')

In [36]:
df_out_with_condition.rename(index={'condition': '#CLASS'}, inplace=True)
df_out_with_condition.head()

,AZ_A8.bam,AZ_C10.bam,AZ_C4.bam,AZ_D6.bam,AZ_E7.bam,AZ_F1.bam,AZ_F2.bam,AZ_H11.bam,HP1502401_A10.bam,HP1502401_A6.bam,...,HP1526901T2D_K7.bam,HP1526901T2D_L1.bam,HP1526901T2D_L5.bam,HP1526901T2D_M15.bam,HP1526901T2D_N1.bam,HP1526901T2D_N12.bam,HP1526901T2D_N13.bam,HP1526901T2D_N23.bam,HP1526901T2D_N3.bam,HP1526901T2D_O14.bam
#CLASS,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,healthy,...,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d,t2d
ATAD3B,0.0,0.0,0.0,0.0,0.073673,0.0,0.0,0.0,0.0,0.0,...,0.767416,0.0,3.377254,0.881244,1.27467,0.0,0.0,0.0,0.23512,0.0
DDX11L17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000228037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PRDM16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.083458,0.0,...,0.0,3.525674,3.454986,0.0,0.0,0.0,1.013528,0.0,0.0,2.307148


In [37]:
df_out_with_condition.to_csv("rpkm_for_GSEA.txt", sep='\t')